<a href="https://colab.research.google.com/github/AntonioAgostini/MNLP-HW1/blob/main/HW1_MNLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Imports section**

*TODO*
- Improve the speed of the running (optimize the number of times the model embed the data)
- compute aggregated metric (e.g. mean(mrr, hit@k))


In [ ]:
import torch
import torch.nn.functional as F
import pprint
import numpy as np
import pandas as pd
from typing import Dict
from random import choice
from datasets import load_dataset
from transformers import EarlyStoppingCallback
from sentence_transformers.evaluation import SentenceEvaluator
from sentence_transformers import (SentenceTransformer,
                                   SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments,
                                   losses)

# **Constants**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 16
lr = 1e-5
epochs = 10
weight_decay = 0.001

print(f"Device: {device}")

Device: cpu


# **Dataset preprocessing**

In [ ]:
# Preparing data for (anchor, positive)
def prepare_pair(example):
    return {
        'anchor': example['query'],
        'positive': example['answer']
    }

# Preparing data for (anchor, positive, negative)
def prepare_triplet(example):
    negatives = [c for c in example["candidate_chunks"] if c != example["answer"]]
    neg = choice(negatives)
    return {
        "anchor": example["query"],
        "positive": example["answer"],
        "negative": neg
    }

ds = load_dataset("sapienzanlp-course-materials/hw-mnlp-2026")

# Splitting the given dataset in 90/10 train/dev
train_dev_split = ds['train'].train_test_split(test_size=0.1, seed=42)
ds_train = train_dev_split['train']
ds_dev = train_dev_split['test']
ds_test = ds['test']

pair_ds_train = ds_train.map(prepare_pair, remove_columns=ds_train.column_names)
pair_ds_dev = ds_dev.map(prepare_pair, remove_columns=ds_dev.column_names)

triplet_ds_train = ds_train.map(prepare_triplet, remove_columns=ds_train.column_names)
triplet_ds_dev = ds_dev.map(prepare_triplet, remove_columns=ds_train.column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

data/blind-00000-of-00001.parquet:   0%|          | 0.00/10.9M [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/66.6M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

# **Evaluation Metrics**

## **Helpers**

In [ ]:
def embed_query_and_candidates(model, query, candidates, batch_size=32):

    q_emb = model.encode(
        [query], # must be a list
        convert_to_tensor = True,
        batch_size = 1,
        show_progress_bar = False,
    )

    # embedding of candidates
    cand_embs = model.encode(
        candidates,
        convert_to_tensor = True,
        batch_size = batch_size,
        show_progress_bar = False,
    )

    return q_emb, cand_embs


### **Hit@k**

Hit@k is a metric used to evaluate the accuracy of retrieval systems. It measures whether the correct answer is present among the top k chunks recommended by the system. If the relevant item (the *answer*) is found within the top k results, it's considered a **hit**.

The formula for Hit@k is:

$$Hit@k = \frac{1}{N} \sum_{i = 1}^{N} \mathbb{I}[rank_i \leq k]$$

where:
- $\mathbb{I}$ is the indicator function;
- $k$ is the number of items taken into consideration.

In [ ]:
def compute_single_hit_at_k(model, k, query, candidates, answer, batch_size=32):

    # query and candidates emebeddings
    q_emb, cand_embs = embed_query_and_candidates(
        model = model,
        query = query,
        candidates = candidates,
    )

    # compute cosine similarity
    scores = F.cosine_similarity(q_emb, cand_embs) # shape [N, dim]

    ranked_idx = torch.argsort(scores, descending=True).tolist()
    correct_idx = candidates.index(answer)
    rank = ranked_idx.index(correct_idx) + 1

    if rank <= k:
      return 1.0

    return 0.0

def compute_hit_at_k(model, dataset, k, batch_size):
    hits = []

    # Calling the hit at k function over each query
    for example in dataset:
        hit = compute_single_hit_at_k(
            model = model,
            k = k,
            query = example["query"],
            candidates = example["candidate_chunks"],
            answer = example["answer"],
            batch_size = batch_size,
        )

        hits.append(hit)

    return float(np.mean(hits))

def compute_multiple_hit_at_k(model, dataset, k_s, batch_size):
    values = {}

    for k in k_s:
        values[k] = compute_hit_at_k(model, dataset, k, batch_size)

    return values

### **MRR**

This metric assesses the effectiveness of a information retrivial system by giving higher scores to systems that rank relevant items higher. It is calculated as the average of the reciprocal ranks of the answer for a set of queries.

The formula for MRR is:

$$MRR = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{rank_i}$$

where:
- $N$ is the total number of queries.
- $rank_i$ is the rank of the first relevant document for the $i$-th query.

In [ ]:
def compute_reciprocal_rank(model, query, candidates, answer, batch_size):

    # query and candidates embeddings
    q_emb, cand_embs = embed_query_and_candidates(
        model = model,
        query = query,
        candidates = candidates,
    )

    # cosine similarity between query and candidate chunks
    scores = F.cosine_similarity(q_emb, cand_embs)

    # Sort in descending order the similarity scores and retrieve
    # the index of the answer to compute 1/rank (reciprocal rank) for
    # the selected query.
    ranked_idx = torch.argsort(scores, descending=True).tolist()
    correct_idx = candidates.index(answer)
    rank = ranked_idx.index(correct_idx) + 1

    return 1.0 / rank

def compute_mrr(model, dataset, batch_size=32):
    reciprocal_ranks = []

    # Calling the reciprocal rank function over each query
    for example in dataset:
        rr = compute_reciprocal_rank(
            model = model,
            query = example["query"],
            candidates = example["candidate_chunks"],
            answer = example["answer"],
            batch_size = batch_size,
        )
        reciprocal_ranks.append(rr)

    return float(np.mean(reciprocal_ranks))

# **Baseline**

In [ ]:
baseline_model = SentenceTransformer("distilbert-base-uncased")
baseline_model.to(device)

baseline_mrr = compute_mrr(
    model = baseline_model,
    dataset = ds_test,
)

k_s = [1, 3, 5]
baseline_hits_at_k = compute_multiple_hit_at_k(
    model = baseline_model,
    dataset = ds_test,
    k_s = k_s,
)

print("Baseline Model: distilbert-base-uncased")
print(f"Baseline MRR: {baseline_mrr}:.4f")
print("Baseline Hit@k:")
print(f"k = 1: {baseline_hits_at_k[1]} | k = 3: {baseline_hits_at_k[3]} | k = 5: {baseline_hits_at_k[5]}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

NameError: name 'device' is not defined

# **Evaluator**

The evaluator measures the performance of the model by computing:
- the MRR.

In [ ]:
class Evaluator(SentenceEvaluator):

    # The batch size is only to improve speed in encoding of chunks
    def __init__(self, dataset, name = None, batch_size=32):
        self.dataset = dataset
        self.batch_size = batch_size
        self.name = name

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1):
        mrr = compute_mrr(
            model = model,
            dataset = self.dataset,
            batch_size = self.batch_size,
        )

        return {"eval_mrr": mrr}

# **Finetuning using triplets**
The finetuning process using triplets involves training the `SentenceTransformer` model to embed sentences such that an 'anchor' (the `query`) sentence is similar in the embedding space to a 'positive' sentence (the `correct answer`) than it is to a 'negative' sentence (the `unrelated answer`). This is achieved through a `TripletLoss` function, which optimizes the model to maintain a certain margin between the anchor-positive and anchor-negative distances.

### **Triplet loss**

*TODO*

In [ ]:

# Instantiate the model
model = SentenceTransformer("distilbert-base-uncased")
model.to(device)

# Instantiate the evaluator
evaluator = Evaluator(ds_dev, batch_size=batch_size)

# Triplet loss for (anchor, positive, negative)
loss = losses.TripletLoss(
    model=model,
    distance_metric=losses.TripletDistanceMetric.COSINE,
    triplet_margin=0.25,
)

# Training arguments
training_args = SentenceTransformerTrainingArguments(
    output_dir = "weights",
    num_train_epochs = epochs,
    per_device_train_batch_size = batch_size,
    per_device_eval_batch_size = batch_size,
    learning_rate = lr,
    weight_decay = weight_decay,

    eval_strategy = "epoch",
    save_strategy = "epoch",
    logging_steps = 50,

    load_best_model_at_end=True,
    metric_for_best_model = "eval_mrr",
    greater_is_better = True,

    warmup_steps = 0.1
)

# Instantiate the trainer
# Notice that the optimizer is not explicitly chosen. By default, AdamW is used.
trainer = SentenceTransformerTrainer(model = model,
                                     args = training_args,
                                     train_dataset = triplet_ds_train,
                                     eval_dataset = triplet_ds_dev,
                                     evaluator = evaluator,
                                     loss = loss,
                                     callbacks = [EarlyStoppingCallback(early_stopping_patience=2)]
                                     )

# Launching the training
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Epoch,Training Loss,Validation Loss,Mrr
1,0.071919,0.069569,0.664081
2,0.048126,0.057095,0.695279
3,0.029463,0.056289,0.693291
4,0.017003,0.053667,0.697928
5,0.008843,0.056130,0.693668
6,0.004478,0.054838,0.695513


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

TrainOutput(global_step=2700, training_loss=0.039760319066268424, metrics={'train_runtime': 3193.0484, 'train_samples_per_second': 22.549, 'train_steps_per_second': 1.409, 'total_flos': 0.0, 'train_loss': 0.039760319066268424, 'epoch': 6.0})